In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 64
learning_rate = 0.01
epochs = 5
num_classes = 10

print(device)


cpu


In [3]:
class VGG(nn.Module):
    def __init__(self, num_classes):
        super(VGG, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.MaxPool2d(kernel_size=2, stride=2)
        )

        self.classifier = nn.Linear(32 * 16 * 16, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [4]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_data = torchvision.datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_data,
    batch_size=batch_size,
    shuffle=False
)


100%|██████████| 170M/170M [36:58<00:00, 76.9kB/s]


In [5]:
model = VGG(num_classes).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=learning_rate,
    momentum=0.9
)

print(model)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU()
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Linear(in_features=8192, out_features=10, bias=True)
)


In [6]:
def train_one_epoch(model, train_loader):
    model.train()

    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


In [7]:
def test_model(model, test_loader):
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            predictions = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predictions == labels).sum().item()

    return correct / total


In [8]:
for epoch in range(epochs):
    loss = train_one_epoch(model, train_loader)
    accuracy = test_model(model, test_loader)

    print(
        "Epoch", epoch + 1,
        "| Loss:", round(loss, 4),
        "| Accuracy:", round(accuracy, 4)
    )


Epoch 1 | Loss: 1.6289 | Accuracy: 0.5194
Epoch 2 | Loss: 1.2027 | Accuracy: 0.6144
Epoch 3 | Loss: 1.0277 | Accuracy: 0.6339
Epoch 4 | Loss: 0.9272 | Accuracy: 0.6419
Epoch 5 | Loss: 0.853 | Accuracy: 0.6565


In [ ]:
torch.save(model.state_dict(), "vgg_block_cifar10.pth")


In [10]:
images, labels = next(iter(test_loader))

images = images.to(device)

outputs = model(images)

predictions = torch.argmax(outputs, dim=1)

print("True labels :", labels[:10].tolist())
print("Predictions :", predictions[:10].cpu().tolist())


True labels : [3, 8, 8, 0, 6, 6, 1, 6, 3, 1]
Predictions : [3, 1, 8, 8, 6, 6, 1, 6, 3, 1]
